# Introduction

This notebook aims at listing many pypowsybl functionalities that can be useful for various analyses:
- define logging and report nodes,
- import data (network, CRAC, parameters),
- launch computations (loadflow, RAO),
- visualise networks and results.

For operational capacity calculation process debugging, the best solution is to retreive the last generated network (in xiidm), so that all preprocessing is done - and also use the crac.json and raoParameters.json (or extract the loadflow parameters from the RAO parameters). Once all input data is gathered, launch a loadflow if the failure occured during a loadflow computation (for instance basecase unsecure), and a RAO if the failure occured during a sensitivity computation (the input data for a sensitivity computation similar to what the RAO does is currently not easy to retreive).

Please see also:
- PyPowSyBl documentation: https://powsybl.readthedocs.io/projects/pypowsybl/en/stable/
- PyPowSyBl notebooks: https://github.com/powsybl/pypowsybl-notebooks
- PyPowSyBl-jupyter (widget to enhance the use of pypowsybl in jupyter by displaying and exploring the network) documentation: https://powsybl.readthedocs.io/projects/pypowsybl-jupyter/en/stable/

In [ ]:
# %pip install pypowsybl
# %pip install pypowsybl-jupyter
# %pip install --upgrade pypowsybl
# %pip install --upgrade pypowsybl-jupyter

# General imports

In [ ]:
home_path = "data/rao/"

network_path = home_path + "12_node_network.xiidm"
crac_path = home_path + "crac_with_monitoring.json"
rao_parameters_path = home_path + "rao_parameters_33.json"

# Define logging level and where to save logs

(1 -> DEBUG -> INFO -> WARNING -> ERROR)

Be careful, if the network import and the computations are run with logging level 1 (or DEBUG?) inside this notebook, its execution might take some time.

https://powsybl.readthedocs.io/projects/pypowsybl/en/stable/user_guide/logging.html

In [ ]:
import logging
from logging import ERROR

logging.basicConfig(filename= home_path + "log.txt",
                    filemode='a', # 'a' for append, 'w' for overwrite
                    format='%(asctime)s - %(levelname)s - %(message)s',
                    datefmt='%Y-%m-%d %H:%M:%S')

# logging.getLogger('powsybl').setLevel(1)
# logging.getLogger('powsybl').setLevel(DEBUG)
logging.getLogger('powsybl').setLevel(ERROR)

# Define report nodes

Alternative/complementary to logs

Not (yet) available for RAO

https://powsybl.readthedocs.io/projects/pypowsybl/en/stable/user_guide/loadflow.html#reports

In [ ]:
import pypowsybl as pp

report_node = pp.report.ReportNode()

# Topology

https://powsybl.readthedocs.io/projects/pypowsybl/en/stable/user_guide/network.html

### Import/export network

For CGMES, zip together all unzipped profiles and boundary sets.

In [ ]:
import pypowsybl as pp

network = pp.network.load(network_path)
# network = pp.network.load(home_path + "network.xiidm")
# network = pp.network.load(home_path + "network.uct")
# network = pp.network.load(home_path + "network.zip") # for CGMES

# network.save(home_path + '/exported_network.xiidm', format='XIIDM')
network.save(home_path + 'exported_network.zip', format='CGMES')

### Basic topological changes

In [ ]:
network.get_generators()
# network.update_generators(id='_1dc9afba-23b5-41a0-8540-b479ed8baf4b', connected=False)
network.get_generators()[['connected','bus_id']]

# Loadflow

https://powsybl.readthedocs.io/projects/pypowsybl/en/stable/user_guide/loadflow.html

### Manually define loadflow parameters

All parameters (and parameters of the extensions) from the json parameter file can be defined this way.

In [ ]:
import pypowsybl.loadflow as lf

loadflow_params = lf.Parameters(
    provider_parameters={'secondaryVoltageControl': 'true',
                         'minPlausibleTargetVoltage': '0.1',
                         'maxPlausibleTargetVoltage': '4',
                         'maxOuterloopIterations': '356789',
                         'maxNewtonRaphsonIterations' : '100'})

### Import RAO parameters and extract loadflow parameters


In [ ]:
from pypowsybl.rao import Parameters as RaoParameters

rao_parameters = RaoParameters.from_file_source(rao_parameters_path)

loadflow_parameters = rao_parameters.search_tree_parameters.loadflow_and_sensitivity_parameters.sensitivity_parameters.load_flow_parameters

#print(loadflow_parameters)

### Export loadflow parameters to JSON file

In [ ]:
import pathlib

export_lf_params_path = home_path + "exportedLoadflowParameters.json"
pathlib.Path(export_lf_params_path).parent.mkdir(parents=True, exist_ok=True)
pathlib.Path(export_lf_params_path).write_text(loadflow_parameters.to_json())

### Run AC and DC loadflow with parameters and report_node

In [ ]:
import pypowsybl.loadflow as lf

lf.run_ac(network, loadflow_parameters, report_node=report_node)
# lf.run_dc(network, parameters, report_node=report_node)

### Retreive report nodes

In [ ]:
print(report_node)
pathlib.Path(home_path + "reportNodes.txt").write_text(str(report_node))

# RAO

https://powsybl.readthedocs.io/projects/pypowsybl/en/stable/user_guide/rao.html

### Modify RAO parameters

In [ ]:
from pypowsybl.rao import Parameters as RaoParameters

rao_parameters = RaoParameters.from_file_source(rao_parameters_path)

rao_parameters.topo_optimization_parameters.max_curative_search_tree_depth = 0
rao_parameters.topo_optimization_parameters.max_preventive_search_tree_depth = 0

### Export raoParameters to file

In [ ]:
import json

export_rao_params_path = home_path + "exportedRaoParameters.json"

with open(export_rao_params_path, "w") as write_file:
    json.dump(rao_parameters.to_json(), write_file, indent=2)

### Import CRAC

In [ ]:
from pypowsybl.rao import Crac

crac = Crac.from_file_source(network, crac_path)

### Filter logs only from OpenRAO

To hide logs from loadflows/sensis/imports inside the RAO computation.

This filter works only with OpenRAO.

In [ ]:
from pypowsybl.rao import RaoLogFilter
from logging import INFO

logging.getLogger('powsybl').setLevel(INFO)
logging.getLogger('powsybl').addFilter(RaoLogFilter())

### Run a RAO computation and read results

Only logging is available for now, but no report nodes.

In [ ]:
rao_runner = pp.rao.create_rao()
rao_result = rao_runner.run(crac=crac, network=network, parameters=rao_parameters)

### Read RAO results

In [ ]:
rao_result.get_flow_cnec_results()
# rao_result.get_remedial_action_results()
# rao_result.get_network_action_results()
# rao_result.get_pst_range_action_results()
# rao_result.get_range_action_results()


### Voltage monitoring

In [ ]:
result_with_voltage_monitoring = rao_runner.run_voltage_monitoring(crac, network, rao_result, provider_str='OpenLoadFlow')

print(result_with_voltage_monitoring.get_voltage_cnec_results())


### Angle monitoring and GLSK import

In [ ]:
from pypowsybl.rao import Glsk as RaoGlsk

monitoring_glsk = RaoGlsk.from_file_source(home_path + "glsk.xml")
result_with_angle_monitoring = rao_runner.run_angle_monitoring(crac, network, rao_result, provider_str='OpenLoadFlow', monitoring_glsk=monitoring_glsk)

print(result_with_angle_monitoring.get_angle_cnec_results())

# Visualisation

https://powsybl.readthedocs.io/projects/pypowsybl/en/stable/user_guide/network_visualization.html

- *network_explorer* is the most complete method for visualisation,
- *get_single_line_diagram* is included in *network_explorer*,
- *get_network_area_diagram* displays the full network, so it is only readable on small networks,
- *write_single_line_diagram_svg* to save the diagram into an svg file.



In [ ]:
from pypowsybl_jupyter import network_explorer

network_explorer(network, depth=1)
# network_explorer(network, depth=1, vl_id='BBE1AA1')
# network.get_single_line_diagram('BBE1AA1')
# network.get_network_area_diagram()

#network.write_single_line_diagram_svg('BBE1AA1', home_path + "single_line_BBE1AA1.svg")